## If on Colab — uncomment the 4 Drive mount lines in Block 2
## If on local CPU — it will warn you about slow training, that's expected

In [ ]:
# ============================================================
# BLOCK 1 — Imports + Device Setup
# ============================================================
# Install transformers if not already installed

import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip',
                       'install', 'transformers', '-q'])

import os
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (BertTokenizer, BertModel,
                          get_linear_schedule_with_warmup)
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (classification_report, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc)

torch.manual_seed(42)
np.random.seed(42)

# ── Device detection — auto selects GPU if available ────────
if torch.cuda.is_available():
    device     = torch.device('cuda')
    device_name = torch.cuda.get_device_name(0)
    USE_AMP    = True    # mixed precision only on GPU
    BATCH_SIZE = 64      # larger batch — GPU has enough memory
    print(f"GPU detected  : {device_name}")
    print(f"VRAM          : "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"Mixed precision: enabled")
    print(f"Batch size     : {BATCH_SIZE}")
else:
    device     = torch.device('cpu')
    USE_AMP    = False   # no mixed precision on CPU
    BATCH_SIZE = 16      # smaller batch — CPU memory constraint
    print(f"No GPU detected — running on CPU")
    print(f"Mixed precision: disabled")
    print(f"Batch size     : {BATCH_SIZE}")
    print(f"Warning        : CPU training will be slow (~3-5 hrs)")

print(f"\nDevice: {device}")

In [ ]:
# ============================================================
# BLOCK 2 — Load Data + Preprocessing
# ============================================================

# ── If running on Colab, mount Drive first ──────────────────
# Uncomment these 4 lines if running on Google Colab
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/fakenews')

# ── Load data ───────────────────────────────────────────────
df_train = pd.read_csv("train.csv")
df_valid = pd.read_csv("valid.csv")
df_test  = pd.read_csv("test.csv")

# ── Drop unnecessary columns ────────────────────────────────
cols_to_drop = ['id', 'date', 'state_info', 'speaker_description']
df_train.drop(columns=cols_to_drop, inplace=True)
df_valid.drop(columns=cols_to_drop, inplace=True)
df_test.drop(columns=cols_to_drop, inplace=True)

# ── Fix nulls ───────────────────────────────────────────────
for df in [df_train, df_valid, df_test]:
    df['subject'] = df['subject'].where(df['subject'].notna(),
                                         other='unknown')
    df['context'] = df['context'].where(df['context'].notna(),
                                         other='unknown')

# ── Binary labels ───────────────────────────────────────────
# {0,1,2} = Fake, {3,4,5} = Real
def to_binary(label):
    return 0 if label in [0, 1, 2] else 1

y_train = df_train['label'].apply(to_binary).values
y_valid = df_valid['label'].apply(to_binary).values
y_test  = df_test['label'].apply(to_binary).values

# ── Speaker credibility ratios + RobustScaler ───────────────
credit_cols = ['true_counts', 'mostly_true_counts', 'half_true_counts',
               'mostly_false_counts', 'false_counts',
               'pants_on_fire_counts']

for df in [df_train, df_valid, df_test]:
    df['total_counts'] = df[credit_cols].sum(axis=1)

ratio_cols = []
for col in credit_cols:
    ratio_col = col.replace('_counts', '_ratio')
    for df in [df_train, df_valid, df_test]:
        df[ratio_col] = df[col] / df['total_counts']
    ratio_cols.append(ratio_col)

scaler       = RobustScaler()
X_train_meta = scaler.fit_transform(df_train[ratio_cols])
X_valid_meta  = scaler.transform(df_valid[ratio_cols])
X_test_meta   = scaler.transform(df_test[ratio_cols])

# ── Raw combined text for BERT ───────────────────────────────
# No cleaning — BERT handles text natively
for df in [df_train, df_valid, df_test]:
    df['raw_combined'] = (df['statement'].fillna('') +
                          ' ' +
                          df['justification'].fillna(''))

# ── Summary ─────────────────────────────────────────────────
print("Data loaded and preprocessed")
print(f"Train : {df_train.shape[0]:,} samples")
print(f"Valid : {df_valid.shape[0]:,} samples")
print(f"Test  : {df_test.shape[0]:,} samples")
print(f"\nBinary distribution (train):")
print(pd.Series(y_train).value_counts())
print(f"\nCredibility matrix shape : {X_train_meta.shape}")
print(f"Ratios sum check         : "
      f"{df_train[ratio_cols].iloc[0].sum().round(3)}")

In [ ]:
# ============================================================
# BLOCK 3 — Tokenization + Dataset + DataLoader
# ============================================================
MAX_LEN   = 256
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_batch(texts, tokenizer, max_len):
    return tokenizer(
        list(texts),
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt',
        add_special_tokens=True
    )

print("Tokenizing train...")
train_encodings = tokenize_batch(df_train['raw_combined'],
                                  tokenizer, MAX_LEN)
print("Tokenizing valid...")
valid_encodings = tokenize_batch(df_valid['raw_combined'],
                                  tokenizer, MAX_LEN)
print("Tokenizing test...")
test_encodings  = tokenize_batch(df_test['raw_combined'],
                                  tokenizer, MAX_LEN)

print(f"\nInput IDs shape     : {train_encodings['input_ids'].shape}")
print(f"Attention mask shape: "
      f"{train_encodings['attention_mask'].shape}")

# ── Dataset ─────────────────────────────────────────────────
class BertFakeNewsDataset(Dataset):
    def __init__(self, encodings, meta_features, labels):
        self.input_ids      = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']
        self.meta_features  = torch.tensor(meta_features,
                                            dtype=torch.float32)
        self.labels         = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids'      : self.input_ids[idx],
            'attention_mask' : self.attention_mask[idx],
            'meta_features'  : self.meta_features[idx],
            'labels'         : self.labels[idx]
        }

train_dataset = BertFakeNewsDataset(train_encodings,
                                     X_train_meta, y_train)
valid_dataset = BertFakeNewsDataset(valid_encodings,
                                     X_valid_meta,  y_valid)
test_dataset  = BertFakeNewsDataset(test_encodings,
                                     X_test_meta,   y_test)

# ── DataLoader ───────────────────────────────────────────────
# num_workers: 2 on GPU (Colab), 0 on CPU (Windows Jupyter)
num_workers = 2 if torch.cuda.is_available() else 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=num_workers,
                          pin_memory=torch.cuda.is_available())
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=num_workers,
                          pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=num_workers,
                          pin_memory=torch.cuda.is_available())

print(f"\nTrain batches : {len(train_loader)}")
print(f"Valid batches : {len(valid_loader)}")
print(f"Test  batches : {len(test_loader)}")

In [ ]:
# ============================================================
# BLOCK 4 — Model Architecture + Training Setup
# ============================================================

# ── Model ───────────────────────────────────────────────────
class BertHybridClassifier(nn.Module):
    def __init__(self, num_classes, meta_dim, dropout=0.3):
        super(BertHybridClassifier, self).__init__()

        # Pretrained BERT base
        self.bert    = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(dropout)

        # 768 (BERT CLS output) + 6 (credibility ratios)
        # → 256 hidden → num_classes
        self.classifier = nn.Sequential(
            nn.Linear(768 + meta_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, meta):
        outputs    = self.bert(input_ids=input_ids,
                               attention_mask=attention_mask)
        cls_output = self.dropout(outputs.pooler_output)
        combined   = torch.cat([cls_output, meta], dim=1)
        return self.classifier(combined)


META_DIM    = 6
NUM_CLASSES = 2
DROPOUT     = 0.3

model_bert = BertHybridClassifier(
    num_classes=NUM_CLASSES,
    meta_dim=META_DIM,
    dropout=DROPOUT
).to(device)

total_params     = sum(p.numel() for p in model_bert.parameters())
trainable_params = sum(p.numel() for p in model_bert.parameters()
                       if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

# ── Training setup ──────────────────────────────────────────
EPOCHS = 4
LR     = 2e-5

class_counts  = np.bincount(y_train)
class_weights = torch.tensor(
    [1.0 / class_counts[0], 1.0 / class_counts[1]],
    dtype=torch.float32
).to(device)
class_weights = class_weights / class_weights.sum()

criterion = nn.CrossEntropyLoss(weight=class_weights)

# AdamW with weight decay — standard for BERT fine-tuning
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model_bert.named_parameters()
                if not any(nd in n for nd in no_decay)],
     'weight_decay': 0.01},
    {'params': [p for n, p in model_bert.named_parameters()
                if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0}
]
optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LR)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Mixed precision scaler — only used if GPU available
scaler_amp = torch.cuda.amp.GradScaler() if USE_AMP else None

PATIENCE         = 2
best_val_loss    = float('inf')
patience_counter = 0
best_model_state = None

print(f"\nEpochs         : {EPOCHS}")
print(f"Learning rate  : {LR}")
print(f"Total steps    : {total_steps}")
print(f"Warmup steps   : {warmup_steps}")
print(f"Class weights  : Fake={class_weights[0]:.4f} | "
      f"Real={class_weights[1]:.4f}")

In [ ]:
# ============================================================
# BLOCK 5 — Training Loop + Evaluation + Visuals + Save
# ============================================================

# ── Train function ──────────────────────────────────────────
def train_epoch_bert(model, loader, optimizer,
                     scheduler, criterion, scaler_amp, use_amp):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        meta           = batch['meta_features'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast('cuda'):
                outputs = model(input_ids, attention_mask, meta)
                loss    = criterion(outputs, labels)
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler_amp.step(optimizer)
            scaler_amp.update()
        else:
            outputs = model(input_ids, attention_mask, meta)
            loss    = criterion(outputs, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()
        total_loss += loss.item()
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


# ── Eval function ───────────────────────────────────────────
def eval_epoch_bert(model, loader, criterion, use_amp):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            meta           = batch['meta_features'].to(device)
            labels         = batch['labels'].to(device)

            if use_amp:
                with torch.amp.autocast('cuda'):
                    outputs = model(input_ids, attention_mask, meta)
                    loss    = criterion(outputs, labels)
            else:
                outputs = model(input_ids, attention_mask, meta)
                loss    = criterion(outputs, labels)

            total_loss += loss.item()
            preds       = outputs.argmax(dim=1)
            probs       = torch.softmax(outputs, dim=1)[:, 1]
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return (total_loss / len(loader), correct / total,
            np.array(all_preds), np.array(all_labels),
            np.array(all_probs))


# ── Training loop ───────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [],
           'train_acc':  [], 'val_acc':  []}

print(f"{'Epoch':<8}{'Train Loss':<14}{'Train Acc':<14}"
      f"{'Val Loss':<14}{'Val Acc':<14}{'Time'}")
print("-" * 72)

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_loss, train_acc = train_epoch_bert(
        model_bert, train_loader, optimizer,
        scheduler, criterion, scaler_amp, USE_AMP)

    val_loss, val_acc, val_preds, val_labels, val_probs = eval_epoch_bert(
        model_bert, valid_loader, criterion, USE_AMP)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - start
    print(f"{epoch:<8}{train_loss:<14.4f}{train_acc:<14.4f}"
          f"{val_loss:<14.4f}{val_acc:<14.4f}{elapsed:.0f}s")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        best_model_state = {k: v.clone() for k, v
                            in model_bert.state_dict().items()}
        print(f"  ✓ Best model saved (val_loss={best_val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print("\nTraining complete.")

# ── Evaluation ──────────────────────────────────────────────
model_bert.load_state_dict(best_model_state)

_, _, val_preds, val_labels, val_probs = eval_epoch_bert(
    model_bert, valid_loader, criterion, USE_AMP)
_, _, test_preds, test_labels, test_probs = eval_epoch_bert(
    model_bert, test_loader, criterion, USE_AMP)

print("\n" + "="*55)
print("VALIDATION RESULTS")
print("="*55)
print(classification_report(val_labels, val_preds,
                             target_names=['Fake', 'Real']))
print("="*55)
print("TEST RESULTS")
print("="*55)
print(classification_report(test_labels, test_preds,
                             target_names=['Fake', 'Real']))

bert_scores = {
    'val_f1'       : f1_score(val_labels,  val_preds,
                               average='weighted'),
    'test_f1'      : f1_score(test_labels, test_preds,
                               average='weighted'),
    'val_f1_macro' : f1_score(val_labels,  val_preds,
                               average='macro'),
    'test_f1_macro': f1_score(test_labels, test_preds,
                               average='macro'),
}
print(f"\nWeighted F1 — Valid: {bert_scores['val_f1']:.4f} | "
      f"Test: {bert_scores['test_f1']:.4f}")
print(f"Macro    F1 — Valid: {bert_scores['val_f1_macro']:.4f} | "
      f"Test: {bert_scores['test_f1_macro']:.4f}")

# ── Save model ──────────────────────────────────────────────
os.makedirs('models', exist_ok=True)

model_bert.bert.save_pretrained('models/bert_base')
tokenizer.save_pretrained('models/bert_tokenizer')

torch.save({
    'classifier_state_dict' : best_model_state,
    'meta_dim'              : META_DIM,
    'num_classes'           : NUM_CLASSES,
    'dropout'               : DROPOUT,
    'max_len'               : MAX_LEN,
    'bert_scores'           : bert_scores,
}, 'models/bert_classifier.pt')

with open('models/robust_scaler_bert.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("\n✓ BERT model saved")
print("✓ Tokenizer saved")
print("✓ Classifier head saved")
print("✓ Scaler saved")

# ── Visuals ─────────────────────────────────────────────────
# Update these with your actual scores from local notebook
svm_scores  = {'val_f1': 0.8617, 'test_f1': 0.8684}
rf_scores   = {'val_f1': 0.8416, 'test_f1': 0.8588}
lstm_scores = {'val_f1': 0.8336, 'test_f1': 0.8521}

fig = plt.figure(figsize=(22, 14))
fig.patch.set_facecolor('#0f0f0f')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Loss curve
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor('#1a1a1a')
epochs_ran = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs_ran, history['train_loss'],
         color='#ff4d6d', lw=2, label='Train Loss')
ax1.plot(epochs_ran, history['val_loss'],
         color='#4dffb4', lw=2, label='Val Loss')
ax1.set_title('BERT Loss Curve', color='white', fontsize=12, pad=12)
ax1.set_xlabel('Epoch', color='white')
ax1.set_ylabel('Loss',  color='white')
ax1.legend(facecolor='#2a2a2a', labelcolor='white')
ax1.tick_params(colors='white')
ax1.spines[:].set_visible(False)

# Accuracy curve
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor('#1a1a1a')
ax2.plot(epochs_ran, history['train_acc'],
         color='#ff4d6d', lw=2, label='Train Acc')
ax2.plot(epochs_ran, history['val_acc'],
         color='#4dffb4', lw=2, label='Val Acc')
ax2.set_title('BERT Accuracy Curve',
              color='white', fontsize=12, pad=12)
ax2.set_xlabel('Epoch', color='white')
ax2.set_ylabel('Accuracy', color='white')
ax2.legend(facecolor='#2a2a2a', labelcolor='white')
ax2.tick_params(colors='white')
ax2.spines[:].set_visible(False)

# ROC curve
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor('#1a1a1a')
fpr_v, tpr_v, _ = roc_curve(val_labels,  val_probs)
fpr_t, tpr_t, _ = roc_curve(test_labels, test_probs)
auc_v = auc(fpr_v, tpr_v)
auc_t = auc(fpr_t, tpr_t)
ax3.plot(fpr_v, tpr_v, color='#4dffb4', lw=2,
         label=f'Validation AUC = {auc_v:.3f}')
ax3.plot(fpr_t, tpr_t, color='#ff4d6d', lw=2,
         label=f'Test AUC = {auc_t:.3f}')
ax3.plot([0,1],[0,1], color='gray', lw=1,
         linestyle='--', label='Random baseline')
ax3.set_title('ROC Curve — BERT',
              color='white', fontsize=12, pad=12)
ax3.set_xlabel('False Positive Rate', color='white')
ax3.set_ylabel('True Positive Rate',  color='white')
ax3.legend(facecolor='#2a2a2a', labelcolor='white', fontsize=9)
ax3.tick_params(colors='white')
ax3.spines[:].set_visible(False)

# Confusion matrix — validation
ax4 = fig.add_subplot(gs[1, 0])
cm_val = confusion_matrix(val_labels, val_preds)
disp   = ConfusionMatrixDisplay(confusion_matrix=cm_val,
                                 display_labels=['Fake', 'Real'])
disp.plot(ax=ax4, colorbar=False, cmap='YlOrRd')
ax4.set_title('Confusion Matrix — Validation',
              color='white', fontsize=12, pad=12)
ax4.set_facecolor('#1a1a1a')
ax4.tick_params(colors='white')
ax4.xaxis.label.set_color('white')
ax4.yaxis.label.set_color('white')

# Confusion matrix — test
ax5 = fig.add_subplot(gs[1, 1])
cm_test = confusion_matrix(test_labels, test_preds)
disp2   = ConfusionMatrixDisplay(confusion_matrix=cm_test,
                                  display_labels=['Fake', 'Real'])
disp2.plot(ax=ax5, colorbar=False, cmap='YlOrRd')
ax5.set_title('Confusion Matrix — Test',
              color='white', fontsize=12, pad=12)
ax5.set_facecolor('#1a1a1a')
ax5.tick_params(colors='white')
ax5.xaxis.label.set_color('white')
ax5.yaxis.label.set_color('white')

# All models comparison
ax6 = fig.add_subplot(gs[1, 2])
ax6.set_facecolor('#1a1a1a')
model_names = ['SVM', 'Random\nForest', 'LSTM', 'BERT']
val_f1s = [svm_scores['val_f1'],  rf_scores['val_f1'],
           lstm_scores['val_f1'], bert_scores['val_f1']]
tst_f1s = [svm_scores['test_f1'], rf_scores['test_f1'],
           lstm_scores['test_f1'], bert_scores['test_f1']]
x     = np.arange(len(model_names))
width = 0.35
ax6.bar(x - width/2, val_f1s, width, color='#4dc9ff',
        label='Validation', edgecolor='none')
ax6.bar(x + width/2, tst_f1s, width, color='#a78bfa',
        label='Test',       edgecolor='none')
for i, (v, t) in enumerate(zip(val_f1s, tst_f1s)):
    ax6.text(i - width/2, v + 0.003, f'{v:.3f}',
             ha='center', fontsize=8, color='white',
             fontweight='bold')
    ax6.text(i + width/2, t + 0.003, f'{t:.3f}',
             ha='center', fontsize=8, color='white',
             fontweight='bold')
ax6.set_xticks(x)
ax6.set_xticklabels(model_names, color='white', fontsize=10)
ax6.set_ylabel('Weighted F1', color='white')
ax6.set_title('All Models Comparison',
              color='white', fontsize=12, pad=12)
ax6.set_ylim(0.75, 1.00)
ax6.legend(facecolor='#2a2a2a', labelcolor='white')
ax6.tick_params(colors='white')
ax6.spines[:].set_visible(False)

plt.suptitle('BERT Fine-tuning Results + Full Model Comparison',
             color='white', fontsize=15, fontweight='bold')
plt.savefig('bert_results.png', dpi=150, bbox_inches='tight',
            facecolor='#0f0f0f')
plt.show()
print("Saved as bert_results.png")